# Frozen ensemble inference

Use a T4 runtime and existing Drive checkpoints. This notebook loads the
RoBERTa checkpoint and the three DeBERTa checkpoints, then applies the frozen
dev-selected ensemble. It does not train or retune models.

Authors: Baltabayev Nuriman and Zhakupov Anvar.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
SOURCE = "upload"  # "github" only after committing the patch
ARCHIVE = Path("/content/mer_experiments_colab.zip")
if SOURCE == "upload":
    if not ARCHIVE.exists():
        from google.colab import files
        uploaded = files.upload()
        name = next((n for n in uploaded if n.endswith(".zip")), None)
        if name is None:
            raise ValueError("Upload dist/mer_experiments_colab.zip")
        ARCHIVE.write_bytes(uploaded[name])
    ROOT = Path("/content") / ("mer-experiments-" + hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()[:10])
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        if sum(i.file_size for i in archive.infolist()) > 20 * 1024**2:
            raise ValueError("Unexpectedly large source archive")
        for info in archive.infolist():
            target = (ROOT / info.filename).resolve()
            if not target.is_relative_to(ROOT.resolve()) or info.filename.startswith(("/", "\\")):
                raise ValueError("Unsafe archive member")
        archive.extractall(ROOT)
    manifest = json.loads((ROOT / "source_manifest.json").read_text())
    for name, expected in manifest["files"].items():
        if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Source hash mismatch: {name}")
else:
    ROOT = Path("/content/mer-experiments-github")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/NurimanB/Multimodal-Emotion-Recognition.git", str(ROOT)], check=True)
    if not (ROOT / "requirements-experiments.txt").exists():
        raise RuntimeError("The experiment patch is not on this GitHub revision. Use SOURCE='upload'.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Source:", ROOT)

## 1. Install the recorded software versions

This preserves Colab's CUDA-enabled PyTorch. If the cell requests a restart, restart the **session** and rerun the setup cells; uploaded files remain on /content. No GitHub token is needed for ZIP upload.

In [ ]:
import importlib.metadata
loaded = {name: getattr(sys.modules[name], "__version__", None)
          for name in ["numpy", "pandas", "transformers"] if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-experiments.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
changed = [name for name, version in loaded.items() if version != importlib.metadata.version(name)]
if changed:
    raise RuntimeError(f"Restart the Colab session and rerun setup: preloaded packages changed: {changed}")
import torch, transformers, pandas as pd
print("torch:", torch.__version__, "transformers:", transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training")
print("GPU:", torch.cuda.get_device_name(0), "| GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)

## Mount Drive and load the frozen four-checkpoint model
The paths use the completed v1 and multiseed_v3 runs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
"""Final frozen four-checkpoint ensemble demo for an already configured Colab."""
from pathlib import Path
import gc
import hashlib
import json
import time

import numpy as np
import torch

from mer.data.meld import EMOTIONS
from mer.evaluation.artifacts import benchmark, write_json
from mer.evaluation.ensemble import blend
from mer.evaluation.inference import Predictor, launch_widget
from mer.evaluation.paths import locate_run

PROJECT_ROOT = Path("/content/drive/MyDrive/diploma-mer")
DEMO_ROOT = PROJECT_ROOT / "experiments"
DEMO_RUN = locate_run(PROJECT_ROOT, "multiseed_v3")
REFERENCE_RUN = locate_run(PROJECT_ROOT, "v1")


class FrozenEnsemblePredictor:
    def __init__(self):
        if not torch.cuda.is_available():
            raise RuntimeError("Use the T4 runtime for the four-checkpoint demo")
        selection_path = DEMO_RUN / "ensemble_roberta_deberta_mean/selection.json"
        self.selection = json.loads(selection_path.read_text())
        guard = json.loads((selection_path.parent / "frozen_protocol.json").read_text())
        if hashlib.sha256(selection_path.read_bytes()).hexdigest() != guard["selection_sha256"]:
            raise ValueError("The frozen selection was changed")
        if self.selection["seeds"] != [42, 123, 2025]:
            raise ValueError("Unexpected ensemble seeds")
        paths = [
            (Path("/content/drive/MyDrive/diploma-mer/outputs/text_roberta_large_ctx3/saved_model"),
             DEMO_RUN / "reused_inputs/roberta_recheck/config.json"),
            (REFERENCE_RUN / "deberta_s42/best_model",
             DEMO_RUN / "reused_inputs/deberta_s42/config.json"),
            (DEMO_RUN / "deberta_s123/best_model", DEMO_RUN / "deberta_s123/config.json"),
            (DEMO_RUN / "deberta_s2025/best_model", DEMO_RUN / "deberta_s2025/config.json"),
        ]
        self.members = []
        for checkpoint, config in paths:
            print("Loading demo checkpoint:", checkpoint, flush=True)
            self.members.append(Predictor(checkpoint, config))
        self.model = self.members[1].model
        self.device = self.members[1].device
        self.parameters = sum(sum(p.numel() for p in member.model.parameters()) for member in self.members)

    def predict(self, text, context="", speaker="Speaker"):
        outputs = [member.predict(text, context, speaker) for member in self.members]
        arrays = [np.array([[output["scores"][emotion] for emotion in EMOTIONS]])
                  for output in outputs]
        probabilities = blend([arrays[0], np.mean(arrays[1:], axis=0)],
                              self.selection["weights"], self.selection["neutral_bias"])[0]
        result = dict(outputs[1])
        result.update(
            emotion=EMOTIONS[int(probabilities.argmax())],
            confidence=float(probabilities.max()),
            scores={emotion: float(p) for emotion, p in zip(EMOTIONS, probabilities)},
            confidence_note="Uncalibrated blended score after neutral bias; tokens shown for DeBERTa seed 42",
            model="Frozen RoBERTa + three-seed DeBERTa ensemble",
            weights=self.selection["weights"], neutral_bias=self.selection["neutral_bias"],
            token_view="DeBERTa-v3-base seed 42; RoBERTa uses a different tokenizer",
        )
        return result


if __name__ == "__main__":
    # Remove the old demo from GPU while loading the final ensemble.
    previous_demo = globals().get("DEMO")
    if previous_demo is not None and hasattr(previous_demo, "model"):
        previous_demo.model.to("cpu")
        gc.collect()
        torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    ENSEMBLE_DEMO = globals().get("ENSEMBLE_DEMO") or FrozenEnsemblePredictor()
    load_seconds = time.perf_counter() - started
    prompts = [
        "I am so happy to see you!",
        "I cannot believe you did that!",
        "The meeting starts at six.",
    ]
    examples = [ENSEMBLE_DEMO.predict(prompt, speaker="Rachel") for prompt in prompts]
    latency = benchmark(
        lambda: ENSEMBLE_DEMO.predict(prompts[0], speaker="Rachel"),
        warmup=3, repeats=30, synchronize=torch.cuda.synchronize,
    )
    evidence = {
        "purpose": "Typed live-demo prompts, not a held-out evaluation",
        "examples": examples,
        "latency": latency,
        "hardware": {
            "gpu": torch.cuda.get_device_name(0),
            "parameters": ENSEMBLE_DEMO.parameters,
            "peak_cuda_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
            "load_seconds": load_seconds,
            "scope": "Four checkpoints simultaneously resident; sequential batch-1 inference; current Colab session",
        },
        "selection": ENSEMBLE_DEMO.selection,
        "confidence_note": "Uncalibrated score; no claim of correctness without ground truth",
    }
    destination = DEMO_ROOT / "inference_demo/demo_evidence.json"
    write_json(destination, evidence)
    print("DEMO_READY", json.dumps({"p50_ms": latency["p50_ms"],
          "p95_ms": latency["p95_ms"], "throughput": latency["throughput_examples_per_second"],
          "first_example": {k: examples[0][k] for k in ["input", "emotion", "confidence", "sequence_length"]}}),
          flush=True)
    launch_widget(ENSEMBLE_DEMO)
    from google.colab import files
    files.download(str(destination))


## Live input
Replace the example with user-provided English text. Context is optional,
one previous turn per line, with speaker names. Do not modify the frozen weights.
The score is uncalibrated; the displayed token view is from DeBERTa seed 42.


In [ ]:
TEXT = "I am so happy to see you!" #@param {type:"string"}
SPEAKER = "Rachel" #@param {type:"string"}
CONTEXT = "" #@param {type:"string"}
result = ENSEMBLE_DEMO.predict(TEXT, context=CONTEXT, speaker=SPEAKER)
print(json.dumps(result, indent=2, ensure_ascii=False))
display(__import__("pandas").DataFrame(result["scores"].items(), columns=["Emotion", "Uncalibrated score"]))
